# HippoVoice — Weight-Editing Baseline (ROME/MEMIT on GPT-2 XL, Kaggle T4)

Benchmarks `baselines/weight_edit_baseline.py`'s real, non-mocked path
against the same LoCoMo QA benchmark HippoVoice/Mem0-style/A-MEM-style/
NaiveRAG already ran through — same harness (`run_locomo`), same scoring
(LoCoMo's real stemmed token-F1), so this is a genuine apples-to-apples
comparison, not an isolated demo.

**What this tests**: instead of storing facts externally and retrieving
them at generation time (every other system in this project), can directly
editing a model's weights with each new fact work as well or better?
Nobody does this for personalized conversational memory in production —
see `baselines/weight_edit_baseline.py`'s module docstring for why — but
this actually checks rather than assuming.

**Expected outcome, stated up front**: ROME/MEMIT were validated on
single, discrete factual edits, not hundreds of accumulating personal
facts over a long conversation. Published work shows sequential edits
degrading well before LoCoMo's 369-689 turns/conversation. Seeing that
degradation for real here is a legitimate result, not a failed benchmark.

**Run this top to bottom once, in order.** Section 3b is a deliberate,
cheap single-edit sanity check before anything else — this project's
established discipline (see `BUGS.md`, the extraction-prompt saga) is to
validate small before spending GPU time on a full run, and this notebook
has real, unresolved uncertainty flagged inline (mainly: EasyEdit's
`keep_original_weight` behavior across repeated single-edit calls, and
whether its `requirements.txt` install fights Kaggle's preinstalled
torch/CUDA) that 3b is specifically designed to catch early, in seconds,
rather than hours into a full LoCoMo run.

## 1. Clone hippovoice + GPU check

In [ ]:
# Same pattern as colab.ipynb's Step 2 -- run after selecting
# Accelerator: GPU T4 x2 in Kaggle's notebook settings (top right).
# Private repo: add a GitHub PAT to Kaggle Secrets as GH_TOKEN, then enable it.

import os, sys

if os.path.exists('/kaggle/working'):
    REPO_DIR = '/kaggle/working/hippovoice'
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GH_TOKEN')
        CLONE_URL = f'https://{token}@github.com/shivansh193/hippovoice.git'
    except Exception:
        CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'
        print('No GH_TOKEN secret found -- will work for public repos only')
else:
    REPO_DIR = '/content/hippovoice'
    CLONE_URL = 'https://github.com/shivansh193/hippovoice.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull
else:
    !git clone {CLONE_URL} {REPO_DIR}

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import subprocess
try:
    commit = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h'], text=True).strip()
except Exception:
    commit = 'unknown'
print(f'commit [{commit}]')
print('Ready.')

In [ ]:
import torch

# Confirmed the hard way on this project's other Kaggle work (see BUGS.md,
# colab.ipynb): checking immediately after clone fails in seconds instead
# of hours into a run that turns out to be running on CPU. ROME/MEMIT edits
# involve real gradient descent (v_num_grad_steps=20 per edit) -- CPU would
# make even the single-edit sanity check in 3b painfully slow, let alone a
# full benchmark.
assert torch.cuda.is_available(), (
    'No GPU detected -- set Accelerator to "GPU T4 x2" in Kaggle Settings '
    '(top-right of the notebook editor) before running anything below.'
)
print(f'GPU OK: {torch.cuda.get_device_name(0)}')

## 2. Clone + install EasyEdit

Not pip-installable in a way that ships its `hparams/*.yaml` files --
`ROMEHyperParams.from_hparams()` reads those directly from a cloned repo,
so cloning is required regardless of whether a PyPI `easyeditor` package
also exists. Installs EasyEdit's own `requirements.txt` (pins `torch==2.9.1`,
`transformers==5.5.4`, and about two dozen other packages -- exact list
confirmed by reading the file directly, not guessed) plus `google-genai`
(this project's own Gemini extraction client, not part of EasyEdit).

**Flagged, not yet run for real**: EasyEdit's `requirements.txt` reinstalling
torch on top of whatever Kaggle's GPU image ships could in principle fight
the preinstalled CUDA build. If imports below fail with a CUDA/torch
version error, the fix is almost certainly `Session → Restart session`
and re-running from here -- but this hasn't been hit yet, so it's a
predicted failure mode, not a confirmed one.

In [ ]:
EASYEDIT_DIR = os.path.join(os.path.dirname(REPO_DIR), 'EasyEdit')

if os.path.exists(os.path.join(EASYEDIT_DIR, '.git')):
    !git -C {EASYEDIT_DIR} pull
else:
    !git clone https://github.com/zjunlp/EasyEdit.git {EASYEDIT_DIR}

!pip install -q -r {EASYEDIT_DIR}/requirements.txt
!pip install -q google-genai

print(f'EasyEdit cloned to {EASYEDIT_DIR}')

## 3. Load GPT-2 XL + ROME (one-time -- reused across conversations)

Loaded once here, not per-conversation: `WeightEditBaseline` resets this
same editor's weights back to pristine at the start of each conversation
(see its `__init__` and the module docstring) rather than reloading the
~6GB model from HuggingFace every time, exactly mirroring how the RAG
baselines reuse one loaded LLM across conversations and only reset their
own memory store per conversation.

In [ ]:
from baselines.easyedit_weight_editor import EasyEditWeightEditor

# ROME, not MEMIT: confirmed directly from both hparams files that ROME's
# gpt2-xl config sets mom2_adjustment: false (no covariance-statistics
# precompute needed) while MEMIT's sets it true with mom2_n_samples: 100000
# -- a real, unmeasured cost this notebook doesn't try to pay. Swap to
# method='MEMIT' once ROME is validated end-to-end, same interface.
editor = EasyEditWeightEditor(easyedit_dir=EASYEDIT_DIR, method='ROME')
print('Loading GPT-2 XL (first run downloads ~6GB from HuggingFace)...')
editor.load()
print('Loaded.')

## 3b. Cheap sanity check -- confirm editing actually works before anything else

Deliberately not a LoCoMo question -- a well-known fact whose default
completion is predictable, so a wrong "BEFORE" or unchanged "AFTER" is
obviously a real problem rather than model uncertainty. **Read the three
printed lines before continuing**: BEFORE should be the true, un-edited
answer; AFTER-edit should reflect the new fact; AFTER-reset should return
to (approximately) the BEFORE answer. If AFTER-edit doesn't change, or
AFTER-reset doesn't revert, stop here and debug -- nothing downstream
(hundreds of sequential edits per LoCoMo conversation) will work either if
a single edit doesn't.

In [ ]:
prompt = "The Eiffel Tower is located in the city of"

print("BEFORE edit: ", editor.generate(prompt, max_tokens=8))

editor.edit(prompt=prompt, subject="The Eiffel Tower", target_new="Rome")
print("AFTER edit:  ", editor.generate(prompt, max_tokens=8))

editor.reset()
print("AFTER reset: ", editor.generate(prompt, max_tokens=8))

## 4. Gemini extraction client

Used only for extraction and edit-request conversion (turning a free-text
memory into a ROME-style prompt/subject/target_new) -- never for QA
answers, which come from the edited model itself. Same reasoning as
`llm/gemini_client.py`'s own docstring: keeps everything except the model
actually being edited API-based, no second heavy local model to load
alongside GPT-2 XL.

Add your key to Kaggle Secrets as `GEMINI_API_KEY` and enable it for this
notebook, or paste it directly below for a quick run (remove it again
afterward if you do).

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
    print('Loaded GEMINI_API_KEY from Kaggle Secrets')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        print('No GEMINI_API_KEY in Kaggle Secrets or the environment -- set one before continuing:')
        print("  os.environ['GEMINI_API_KEY'] = '...'")

from llm.gemini_client import GeminiTextLLM
extraction_llm = GeminiTextLLM()
print(f'Extraction LLM: {extraction_llm.model_name}')

## 5. Run WeightEditBaseline through the real LoCoMo benchmark

Same `run_locomo` harness, same F1 scoring, as every other system's LoCoMo
number in this project. `pipeline_factory` closes over the single shared
`editor` loaded in Section 3 -- `WeightEditBaseline.__init__` resets it to
pristine weights for every new conversation (validated locally against
`MockWeightEditor` in `tests/test_weight_edit_baseline.py`; Section 3b just
confirmed `editor.reset()` itself works against the real model too).

**Scope starts deliberately small.** Every extracted, editable memory is a
real ROME edit -- gradient descent against the actual model, not a cheap
dict write -- so per-turn cost here is categorically different from the
RAG baselines' pure-retrieval QA step. `NUM_CONVERSATIONS=1` and
`MAX_QA_PER_CONVERSATION=15` first, to get a real wall-clock/turn reading
before committing to anything close to the other baselines' full
10-conversation, ~1540-question runs. Widen both only after seeing how
long one conversation actually takes.

In [ ]:
from benchmarks.locomo.evaluate import run_locomo
from baselines.weight_edit_baseline import WeightEditBaseline

NUM_CONVERSATIONS = 1
MAX_QA_PER_CONVERSATION = 15

def _weight_edit_factory(llm):
    return WeightEditBaseline(llm_client=llm, editor=editor)

CHECKPOINT_BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
CHECKPOINT_PATH = f'{CHECKPOINT_BASE}/locomo_checkpoint_weightedit_rome.json'

print(f'Running LoCoMo for WeightEdit-ROME '
      f'({NUM_CONVERSATIONS} conversation(s), up to {MAX_QA_PER_CONVERSATION} QA pairs each)...')
print(f'Checkpoint path: {CHECKPOINT_PATH}\n')

import time
_t0 = time.time()
locomo_result = run_locomo(
    llm_client=extraction_llm,
    num_conversations=NUM_CONVERSATIONS,
    max_qa_per_conversation=MAX_QA_PER_CONVERSATION,
    checkpoint_path=CHECKPOINT_PATH,
    pipeline_factory=_weight_edit_factory,
    system_name='WeightEdit-ROME',
)
_elapsed = time.time() - _t0

print('=' * 60)
print(f"WeightEdit-ROME LoCoMo avg F1: {locomo_result['avg_f1']:.1%}  (over {locomo_result['total']} questions)")
print(f"bins: {locomo_result['bins']}")
print(f'Wall clock: {_elapsed:.0f}s for {NUM_CONVERSATIONS} conversation(s) '
      f'-- use this to size NUM_CONVERSATIONS/MAX_QA_PER_CONVERSATION for a bigger run.')
print('=' * 60)
for d in sorted(locomo_result['details'], key=lambda d: -d['f1'])[:10]:
    print(f"  f1={d['f1']:.2f} cat={d['category']}  Q: {d['question']}")
    print(f"    gold={d['gold']!r}  predicted={d['predicted']!r}")

## 6. Save results

In [ ]:
import json, datetime

RESULTS_BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
RESULTS_PATH = f'{RESULTS_BASE}/hippovoice_results_weightedit_rome.json'

out = {
    'timestamp': datetime.datetime.now().isoformat(),
    'gpu': torch.cuda.get_device_name(0),
    'system': 'WeightEdit-ROME',
    'extraction_llm': extraction_llm.model_name,
    'num_conversations': NUM_CONVERSATIONS,
    'max_qa_per_conversation': MAX_QA_PER_CONVERSATION,
    'wall_clock_seconds': _elapsed,
    'locomo': {
        'avg_f1': locomo_result['avg_f1'],
        'total': locomo_result['total'],
        'bins': locomo_result['bins'],
    },
}

with open(RESULTS_PATH, 'w') as f:
    json.dump(out, f, indent=2)

print(f'Saved to {RESULTS_PATH} (visible in the Output tab after commit)')
print(json.dumps(out, indent=2))